# 05. Look-Alike Scoring

> **Краткое резюме**: Определяем эталонную группу (лучшие привлечённые клиенты), обучаем GBM-классификатор «эталон vs остальные», дополняем kNN cosine similarity и объединяем в ансамбль. Результат — ранжированный список проспектов с интерпретируемым `lookalike_score`.

**Алгоритм**:
1. **Эталон**: `clientcounterparty_flag = 'Y'` + `clientchange_date IS NOT NULL` + топ 20% по обороту → **1 830 клиентов**
2. **kNN cosine** (35% веса): средняя похожесть к K ближайшим эталонным соседям
3. **GBM classifier** (65% веса): P(принадлежность к эталону) — нелинейная модель с feature importance
4. **Ensemble**: взвешенное среднее двух сигналов → `lookalike_score ∈ [0, 1]`
5. **Ранжирование**: проспекты (`flag = 'N'`) отсортированы по финальному score

**Почему ансамбль лучше одного центроида:**
- Центроид усредняет 1 830 клиентов → одна точка, которая может быть в малоплотной области
- kNN учитывает форму и плотность облака эталонов
- GBM находит нелинейные паттерны и взаимодействия признаков

**Предпосылки**: `03_feature_engineering` и `04_segmentation` выполнены.

---

In [ ]:
import os
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=UserWarning)

OUTPUT_DIR = os.path.abspath("./data")

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

# Доля лучших клиентов для эталона (топ N% по обороту среди привлечённых)
TOP_PERCENTILE = 0.20  # 20%

# Количество ближайших соседей для kNN-скоринга
K_NEIGHBORS = 50

# Веса в ансамбле: GBM (основной) + kNN cosine (вспомогательный)
WEIGHT_GBM = 0.65
WEIGHT_KNN = 0.35

# Минимальный порог финального score для «горячих» проспектов
SCORE_THRESHOLD = 0.60

# Количество топ-проспектов для экспорта
TOP_N_PROSPECTS = 100

---
## 1. Загрузка данных

In [ ]:
X = pd.read_parquet(os.path.join(OUTPUT_DIR, "feature_matrix.parquet"))
full_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "segments.parquet"))

with open(os.path.join(OUTPUT_DIR, "feature_meta.pkl"), "rb") as f:
    meta = pickle.load(f)

print(f"Feature matrix: {X.shape}")
print(f"Full data: {full_df.shape}")
print("\nDistribution of clientcounterparty_flag:")
print(full_df["clientcounterparty_flag"].value_counts())

---
## 2. Определение эталонной группы

**Лучшие привлечённые клиенты** = те, кто:
1. Сейчас клиент (`clientcounterparty_flag = 'Y'`)
2. Когда-то был контрагентом (`clientchange_date IS NOT NULL`)
3. Имеет высокий оборот (топ N% среди привлечённых)

Если `clientchange_date` не заполнен — используем всех клиентов с высоким оборотом.

In [ ]:
# Клиенты
clients_mask = full_df["clientcounterparty_flag"] == "Y"
print(f"Clients (Y): {clients_mask.sum():,}")

# Привлечённые (были контрагентами → стали клиентами)
if "clientchange_date" in full_df.columns:
    acquired_mask = clients_mask & full_df["clientchange_date"].notna()
    print(f"Acquired (Y + clientchange_date): {acquired_mask.sum():,}")
else:
    acquired_mask = clients_mask
    print("clientchange_date not available — using all clients")

# Если мало привлечённых, используем всех клиентов
if acquired_mask.sum() < 100:
    print(f"Too few acquired ({acquired_mask.sum()}), falling back to all clients")
    acquired_mask = clients_mask

# Топ по обороту среди привлечённых
acquired_df = full_df[acquired_mask].copy()
turnover_threshold = acquired_df["total_amount"].quantile(1 - TOP_PERCENTILE)
reference_mask = acquired_mask & (full_df["total_amount"] >= turnover_threshold)

print(f"\nReference group (top {TOP_PERCENTILE * 100:.0f}% by turnover): {reference_mask.sum():,}")
print(f"Turnover threshold: {turnover_threshold:,.0f}")

---
## 3. Скоринг: kNN + GBM Ensemble

**Два независимых сигнала, объединённых в ансамбль:**
- **kNN cosine** (35%): средняя косинусная похожесть к ближайшим K эталонным клиентам. Улавливает «форму» облака эталонов.
- **GBM** (65%): бинарный классификатор эталон vs. остальные клиенты. Учитывает нелинейные взаимодействия признаков.

Ансамбль устойчивее каждого компонента по отдельности.

In [ ]:
# ── Шаг A: kNN cosine scoring ───────────────────────────────────────────────
X_reference = X.loc[X.index.isin(full_df[reference_mask].index)]
X_reference_vals = X_reference.values  # (n_ref, n_features)
X_all_vals = X.values  # (n_all, n_features)

print(f"Reference vectors: {X_reference_vals.shape[0]:,}")
print(
    f"Computing cosine similarity matrix ({X_all_vals.shape[0]} × {X_reference_vals.shape[0]})..."
)

# sim_matrix: (n_all, n_ref)
sim_matrix = cosine_similarity(X_all_vals, X_reference_vals)

# kNN score = среднее по топ-K ближайшим эталонным соседям
k = min(K_NEIGHBORS, X_reference_vals.shape[0])
knn_scores_raw = np.sort(sim_matrix, axis=1)[:, -k:].mean(axis=1)

# Нормализуем в [0, 1]
knn_min, knn_max = knn_scores_raw.min(), knn_scores_raw.max()
knn_scores = (knn_scores_raw - knn_min) / (knn_max - knn_min + 1e-9)
full_df["score_knn"] = knn_scores

print(f"\nkNN score distribution (k={k}):")
print(pd.Series(knn_scores).describe().round(4))

# ── Шаг B: GBM binary classifier ────────────────────────────────────────────
# Эталон отобран по total_amount >= 80-го перцентиля.
# Любой признак, монотонно связанный с total_amount, даст утечку целевой
# переменной: total_outflow, total_inflow, avg_monthly_amount, avg_tx_amount,
# std_tx_amount, amt_per_cp, avg_balance и т.д.
#
# WHITELIST: GBM обучается только на поведенческих признаках — активность,
# диверсификация контрагентов, динамика, структура, отраслевые dummy.
# kNN использует все фичи — там утечки нет (similarity, не classifier).

GBM_BEHAVIORAL_PREFIXES = (
    "tx_count",  # количество транзакций
    "unique_",  # unique_counterparties, unique_payees, unique_payers
    "active_months",  # регулярность присутствия
    "direction_ratio",  # характер потоков (расходы/доходы)
    "amount_growth",  # темп роста оборота (ratio)
    "cp_growth",  # темп роста числа контрагентов
    "tx_amount_cv",  # вариативность сумм транзакций
    "tx_per_month",  # интенсивность транзакций (tx_count / active_months)
    "cp_per_month",  # интенсивность контрагентов
    "payee_ratio",  # доля получателей среди контрагентов
    "payer_ratio",  # доля плательщиков среди контрагентов
    "product_count",  # число продуктов
    "product_type_count",  # разнообразие типов продуктов
    "okved_",  # отрасль (dummy)
    "region_",  # регион (dummy)
    "mseg_",  # модельный сегмент (dummy)
    "ctype_",  # тип клиента (dummy)
)

gbm_feature_cols = [
    c for c in X.columns if any(c == p or c.startswith(p) for p in GBM_BEHAVIORAL_PREFIXES)
]
excluded = [c for c in X.columns if c not in gbm_feature_cols]
print(f"\nGBM features (behavioral whitelist): {len(gbm_feature_cols)}")
print(f"  Excluded (monetary proxies): {excluded}")

pos_idx = full_df[reference_mask].index
neg_idx = full_df[clients_mask & ~reference_mask].index

X_pos = X.loc[X.index.isin(pos_idx), gbm_feature_cols].values
X_neg = X.loc[X.index.isin(neg_idx), gbm_feature_cols].values
X_train = np.vstack([X_pos, X_neg])
y_train = np.concatenate([np.ones(len(X_pos)), np.zeros(len(X_neg))])

X_all_gbm = X[gbm_feature_cols].values

print(f"GBM training: {len(X_pos):,} positives (reference) vs {len(X_neg):,} negatives")

# Пробуем XGBoost — если недоступен, используем sklearn GBM с early stopping
try:
    from xgboost import XGBClassifier

    clf = XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=len(X_neg) / len(X_pos),
        early_stopping_rounds=30,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
    )
    clf_name = "XGBoost"
    # XGBoost требует eval_set для early stopping
    split = int(len(X_train) * 0.85)
    clf.fit(
        X_train[:split],
        y_train[:split],
        eval_set=[(X_train[split:], y_train[split:])],
        verbose=False,
    )
except ImportError:
    # sklearn GBM с early stopping (доступно с sklearn ≥ 0.20)
    clf = GradientBoostingClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        validation_fraction=0.15,
        n_iter_no_change=30,
        tol=1e-4,
        random_state=42,
    )
    clf_name = "GradientBoosting (sklearn)"
    clf.fit(X_train, y_train)

n_trees = getattr(clf, "n_estimators_", getattr(clf, "n_estimators", "?"))
print(f"{clf_name} trained ({n_trees} trees used).")

gbm_proba = clf.predict_proba(X_all_gbm)[:, 1]
full_df["score_gbm"] = gbm_proba
print(f"Train P(ref) среди эталона:           {gbm_proba[X.index.isin(pos_idx)].mean():.3f}")
print(f"Train P(ref) среди остальных клиентов: {gbm_proba[X.index.isin(neg_idx)].mean():.3f}")

# ── Шаг C: Ensemble ─────────────────────────────────────────────────────────
gbm_norm = (gbm_proba - gbm_proba.min()) / (gbm_proba.max() - gbm_proba.min() + 1e-9)
full_df["lookalike_score"] = WEIGHT_GBM * gbm_norm + WEIGHT_KNN * knn_scores

print(f"\n✅ Финальный score (ансамбль {WEIGHT_GBM:.0%} GBM + {WEIGHT_KNN:.0%} kNN):")
print(full_df["lookalike_score"].describe().round(4))

In [ ]:
# ruff: noqa: F821
if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
else:
    importances = np.zeros(len(gbm_feature_cols))

fi = pd.Series(importances, index=gbm_feature_cols).sort_values(ascending=False)
top_fi = fi.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors_fi = ["#d62728" if i < 5 else "#1f77b4" for i in range(len(top_fi))]
top_fi.plot(kind="barh", ax=ax, color=colors_fi[::-1], edgecolor="black", alpha=0.8)
ax.invert_yaxis()
ax.set_xlabel("Feature Importance (impurity)")
ax.set_title("Топ-20 признаков: что определяет «лучшего клиента»?", fontsize=12)
ax.axvline(0, color="black", linewidth=0.5)

for i, (_name, val) in enumerate(top_fi.items()):
    ax.text(val + 0.001, i, f"{val:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("\n📊 ТОП-5 ПРИЗНАКОВ — ИНТЕРПРЕТАЦИЯ:")
print("─" * 55)
FEATURE_DESCRIPTIONS = {
    "tx_count": "Количество транзакций — мера активности",
    "total_amount": "Суммарный оборот — размер клиента",
    "unique_counterparties": "Число уникальных контрагентов — широта связей",
    "direction_ratio": "Доля расходов — характер потоков",
    "active_months": "Активные месяцы — регулярность",
    "amount_growth": "Рост оборота — динамика развития",
    "cp_growth": "Рост числа контрагентов — расширение",
    "tx_amount_cv": "Вариация сумм транзакций — однородность",
    "avg_monthly_amount": "Средний месячный оборот",
    "avg_tx_amount": "Средняя сумма транзакции",
}
for rank, (fname, fval) in enumerate(top_fi.head(5).items(), 1):
    desc = FEATURE_DESCRIPTIONS.get(fname, fname)
    pct = fval / fi.sum() * 100
    print(f"  #{rank}: {fname:30s} {pct:5.1f}%  ← {desc}")

# Сохраняем feature_names для pickle
feature_names = gbm_feature_cols

---
## 3.5. Важность признаков (Feature Importance)

Какие признаки GBM использует для разделения «лучших клиентов» от остальных?
Это говорит нам: **что именно делает клиента похожим на эталон**.

---
## 4. Ранжирование проспектов

In [ ]:
# ruff: noqa: F821
prospects_mask = full_df["clientcounterparty_flag"].isin(["N", "?"])
prospects = full_df[prospects_mask].sort_values("lookalike_score", ascending=False)

print(f"Total prospects: {len(prospects):,}")
print(
    f"Prospects with score >= {SCORE_THRESHOLD}: "
    f"{(prospects['lookalike_score'] >= SCORE_THRESHOLD).sum():,}"
)

display_cols = [
    "client_name",
    "client_type_name",
    "clientcounterparty_flag",
    "sparkokved_ccode",
    "addrref_city_name",
    "srvpackagesegment_ccode",
    "behavioral_segment",
    "total_amount",
    "unique_counterparties",
    "direction_ratio",
    "lookalike_score",
]
display_cols = [c for c in display_cols if c in prospects.columns]

print(f"\nTop-{TOP_N_PROSPECTS} prospects by lookalike_score:")
display(prospects[display_cols].head(TOP_N_PROSPECTS))

In [ ]:
# ruff: noqa: F821
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, mask in [("Clients (Y)", clients_mask), ("Prospects (N/?)", prospects_mask)]:
    subset = full_df.loc[mask, "lookalike_score"].dropna()
    axes[0].hist(subset, bins=50, alpha=0.5, label=f"{label} ({len(subset):,})", edgecolor="black")
axes[0].axvline(SCORE_THRESHOLD, color="red", linestyle="--", label=f"Threshold {SCORE_THRESHOLD}")
axes[0].set_xlabel("Lookalike Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score Distribution: Clients vs Prospects")
axes[0].legend()

top_prospects = prospects.head(TOP_N_PROSPECTS)
if "behavioral_segment" in top_prospects.columns:
    seg_counts = top_prospects["behavioral_segment"].value_counts().sort_index()
    axes[1].bar(seg_counts.index.astype(str), seg_counts.values, edgecolor="black", alpha=0.7)
    axes[1].set_xlabel("Behavioral Segment")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Top-{TOP_N_PROSPECTS} Prospects by Segment")

plt.tight_layout()
plt.show()

---
## 5. Анализ: профиль top-проспектов vs эталон

In [ ]:
# ruff: noqa: F821
compare_cols = [
    "total_amount",
    "tx_count",
    "unique_counterparties",
    "direction_ratio",
    "active_months",
    "amount_growth",
]
for c in ["tx_per_month", "cp_per_month", "payee_ratio", "avg_balance", "product_count"]:
    if c in full_df.columns:
        compare_cols.append(c)
compare_cols = [c for c in compare_cols if c in full_df.columns]

# compare: index=compare_cols (фичи), columns=группы
compare = pd.DataFrame(
    {
        "Reference (best clients)": full_df.loc[reference_mask, compare_cols].mean(),
        f"Top-{TOP_N_PROSPECTS} prospects": top_prospects[compare_cols].mean(),
        "All prospects": prospects[compare_cols].mean(),
    }
)

# Убираем строки где все значения NaN (нет данных в источнике — напр. product_count)
# Это предотвращает RuntimeWarning: All-NaN slice encountered в MinMaxScaler
compare = compare.loc[~compare.isnull().all(axis=1)]

print("Profile comparison:")
display(compare.round(2))

norm_compare = MinMaxScaler().fit_transform(compare.T)
norm_compare_df = pd.DataFrame(norm_compare, index=compare.columns, columns=compare.index)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bar_cols = [c for c in compare.index[:6]]
x = np.arange(len(bar_cols))
width = 0.28
ref_vals = compare.loc[bar_cols, "Reference (best clients)"]
top_vals = compare.loc[bar_cols, f"Top-{TOP_N_PROSPECTS} prospects"]
all_vals = compare.loc[bar_cols, "All prospects"]

ref_norm = ref_vals / ref_vals.replace(0, np.nan)
top_norm = top_vals / ref_vals.replace(0, np.nan)
all_norm = all_vals / ref_vals.replace(0, np.nan)

axes[0].bar(
    x - width, ref_norm, width, label="Reference", color="#2ca02c", alpha=0.85, edgecolor="black"
)
axes[0].bar(
    x,
    top_norm,
    width,
    label=f"Top-{TOP_N_PROSPECTS}",
    color="#1f77b4",
    alpha=0.85,
    edgecolor="black",
)
axes[0].bar(
    x + width,
    all_norm,
    width,
    label="All prospects",
    color="#ff7f0e",
    alpha=0.85,
    edgecolor="black",
)
axes[0].axhline(1.0, color="green", linestyle="--", alpha=0.5, label="Эталон = 1.0")
axes[0].set_xticks(x)
axes[0].set_xticklabels([c[:12] for c in bar_cols], rotation=30, ha="right")
axes[0].set_ylabel("Относительно эталона")
axes[0].set_title("Профиль: топ-проспекты vs эталон (нормировано)")
axes[0].legend(fontsize=8)

radar_cols = list(compare.index[: min(7, len(compare.index))])
angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]
ax_r = axes[1]
ax_r.remove()
ax_r = fig.add_subplot(1, 2, 2, polar=True)

for i, (group, row) in enumerate(norm_compare_df[radar_cols].iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    color = ["#2ca02c", "#1f77b4", "#ff7f0e"][i % 3]
    ax_r.plot(angles, vals, "o-", label=group, color=color, linewidth=2)
    ax_r.fill(angles, vals, alpha=0.07, color=color)
ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels([c[:10] for c in radar_cols], fontsize=8)
ax_r.set_title("Радар-профиль", fontsize=11, pad=15)
ax_r.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=8)

plt.suptitle("Профиль top-проспектов vs эталон vs все проспекты", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 ИНТЕРПРЕТАЦИЯ ПРОФИЛЯ:")
print("─" * 60)
for col in compare.index:
    ref_v = compare.loc[col, "Reference (best clients)"]
    top_v = compare.loc[col, f"Top-{TOP_N_PROSPECTS} prospects"]
    if pd.isna(ref_v) or ref_v == 0:
        continue
    ratio = top_v / ref_v
    trend = "✅" if ratio >= 0.7 else ("⚠️" if ratio >= 0.4 else "❌")
    print(
        f"  {trend} {col:28s}: top-{TOP_N_PROSPECTS} = {top_v:8.2f}  "
        f"(эталон = {ref_v:.2f}, покрытие {ratio:.0%})"
    )

---
## 6. Lift-анализ

Насколько модель лучше случайного выбора?

In [ ]:
# ruff: noqa: F821
if "clientchange_date" in full_df.columns and acquired_mask.sum() > 0:
    ranked = full_df.sort_values("lookalike_score", ascending=False).copy()
    ranked["rank_pct"] = np.arange(1, len(ranked) + 1) / len(ranked)
    ranked["is_acquired"] = ranked.index.isin(full_df[acquired_mask].index)

    pct_points = [0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50]
    lifts = []
    for pct in pct_points:
        top_k = ranked[ranked["rank_pct"] <= pct]
        acquired_in_top = int(top_k["is_acquired"].sum())
        expected = acquired_mask.sum() * pct
        lift = acquired_in_top / max(expected, 1)
        lifts.append(
            {
                "top_%": f"{pct * 100:.0f}%",
                "acquired_found": acquired_in_top,
                "expected_random": round(expected),
                "lift": round(lift, 2),
                "lift_str": f"{lift:.2f}x",
            }
        )

    lift_df = pd.DataFrame(lifts)
    print("Lift analysis (acquired clients in top-K% by score):")
    display(lift_df[["top_%", "acquired_found", "expected_random", "lift_str"]])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    xs = [float(row["top_%"].strip("%")) for row in lifts]
    lift_vals = [row["lift"] for row in lifts]

    axes[0].plot(xs, lift_vals, "o-", linewidth=2, color="#1f77b4", markersize=8)
    axes[0].axhline(1.0, color="gray", linestyle="--", alpha=0.7, label="Random baseline")
    for x_pt, y_pt in zip(xs, lift_vals, strict=False):
        axes[0].annotate(
            f"{y_pt:.2f}x",
            (x_pt, y_pt),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=9,
        )
    axes[0].set_xlabel("Top % by score")
    axes[0].set_ylabel("Lift")
    axes[0].set_title("Lift Curve: модель vs случайный выбор")
    axes[0].legend()
    axes[0].set_ylim(0.8, max(lift_vals) * 1.2)

    found_vals = [row["acquired_found"] for row in lifts]
    total_acquired = acquired_mask.sum()
    gain_pct = [v / total_acquired * 100 for v in found_vals]

    axes[1].plot(xs, gain_pct, "o-", linewidth=2, color="#2ca02c", label="Модель", markersize=8)
    axes[1].plot([0] + xs, [0] + xs, "--", color="gray", label="Random baseline")
    axes[1].fill_between(xs, gain_pct, xs[: len(xs)], alpha=0.15, color="#2ca02c")
    axes[1].set_xlabel("Top % отобрано")
    axes[1].set_ylabel("% привлечённых клиентов найдено")
    axes[1].set_title("Cumulative Gain Curve")
    axes[1].legend()

    plt.suptitle("Качество ранжирования модели", fontsize=13)
    plt.tight_layout()
    plt.show()

    top5_lift = next(row["lift"] for row in lifts if row["top_%"] == "5%")
    top10_lift = next(row["lift"] for row in lifts if row["top_%"] == "10%")
    top5_gain = next(row["acquired_found"] for row in lifts if row["top_%"] == "5%")

    print("\n📊 ИНТЕРПРЕТАЦИЯ LIFT-АНАЛИЗА:")
    print("─" * 60)
    print(f"  • Модель в топ-5% находит в {top5_lift:.2f}x больше привлечённых,")
    print(
        f"    чем случайный выбор ({top5_gain:,} vs {int(acquired_mask.sum() * 0.05):,} ожидаемых)"
    )
    print(f"  • Lift {top10_lift:.2f}x в топ-10% — модель эффективно отделяет")
    print("    «конвертируемых» контрагентов от остальных")
    if top5_lift > 1.5:
        print("  • ✅ Хороший lift: модель значительно лучше случайного выбора")
    elif top5_lift > 1.2:
        print("  • ⚠️  Умеренный lift: работает лучше random, но есть куда расти")
    else:
        print("  • ❌  Низкий lift: модель почти не лучше случайного выбора")
else:
    print("clientchange_date not available — lift analysis skipped.")

---
## 7. Сохранение результатов

In [ ]:
# ruff: noqa: F821
scores_path = os.path.join(OUTPUT_DIR, "lookalike_scores.parquet")
full_df.to_parquet(scores_path)
print(f"Full scores saved: {scores_path}")

model_path = os.path.join(OUTPUT_DIR, "lookalike_gbm.pkl")
with open(model_path, "wb") as f:
    pickle.dump({"model": clf, "model_name": clf_name, "feature_names": feature_names}, f)
print(f"GBM model saved: {model_path}")

export_cols = [
    c
    for c in [
        "client_name",
        "client_type_name",
        "clientcounterparty_flag",
        "sparkokved_ccode",
        "addrref_city_name",
        "reg_city_name",
        "srvpackagesegment_ccode",
        "behavioral_segment",
        "segment_label",
        "total_amount",
        "tx_count",
        "unique_counterparties",
        "direction_ratio",
        "active_months",
        "amount_growth",
        "score_knn",
        "score_gbm",
        "lookalike_score",
    ]
    if c in prospects.columns
]

top_path = os.path.join(OUTPUT_DIR, "top_prospects.parquet")
prospects[export_cols].head(TOP_N_PROSPECTS * 5).to_parquet(top_path)
print(f"Top prospects saved: {top_path}")

try:
    xlsx_path = os.path.join(OUTPUT_DIR, "top_prospects.xlsx")
    (
        prospects[export_cols]
        .head(TOP_N_PROSPECTS * 5)
        .rename(
            columns={
                "lookalike_score": "score_final",
                "sparkokved_ccode": "okved",
                "addrref_city_name": "city",
                "srvpackagesegment_ccode": "model_segment",
            }
        )
        .to_excel(xlsx_path, index=True)
    )
    print(f"Excel saved: {xlsx_path}")
except Exception as e:
    print(f"Excel export skipped ({e})")

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **Эталонная группа** | Лучшие привлечённые клиенты: были контрагентами → стали клиентами + топ 20% по обороту |
| **kNN cosine** | Среднее cosine similarity к K ближайшим эталонным клиентам. Учитывает форму облака |
| **GBM** | Gradient Boosting Machine — ансамбль деревьев. P(эталон) как score |
| **XGBoost** | Оптимизированный GBM. Используется если установлен, иначе sklearn GBM |
| **Ensemble score** | 0.65 × GBM + 0.35 × kNN, нормализованный в [0, 1] |
| **Feature Importance** | Какие признаки GBM использует для разделения классов (impurity-based) |
| **Lift** | Во сколько раз модель лучше случайного выбора в топ-K% |
| **Cumulative Gain** | Какую долю привлечённых клиентов мы найдём, отобрав X% базы |
| **clientchange_date** | Дата перехода из контрагентов в клиенты — маркер «конвертированного» |
| **score_knn / score_gbm** | Промежуточные score компонентов (для отладки и объяснения) |

---

**Pipeline завершён.** Файлы в `data/`:
- `lookalike_scores.parquet` — все клиенты со scores
- `top_prospects.parquet` / `.xlsx` — топ-500 проспектов для менеджеров
- `lookalike_gbm.pkl` — обученная GBM модель